### Lancedb - a vector database for LLM applications

In [8]:
import lancedb 

db = lancedb.connect(uri = "vector_database")
db

LanceDBConnection(uri='/Users/efrem_27/Documents/github/AI_engineering/09_lancedb_vector_database/vector_database')

In [11]:
db.uri

'/Users/efrem_27/Documents/github/AI_engineering/09_lancedb_vector_database/vector_database'

### Create a table

In [12]:
import json 

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [13]:
table_name = db.create_table("animals_text", exist_ok= True, data=data)
table_name

LanceTable(name='animals_text', version=2, _conn=LanceDBConnection(uri='/Users/efrem_27/Documents/github/AI_engineering/09_lancedb_vector_database/vector_database'))

In [14]:
table_name.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [15]:
more_data = [ {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
              {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

table_name.add(more_data)

AddResult(version=3)

In [16]:
table_name.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


### Create an empty tbale and then delete it 

In [17]:
from lancedb.pydantic import LanceModel

class JokeSchema(LanceModel):
    joke: str
    rating: int

db.create_table(name="Jokes", schema=JokeSchema, exist_ok=True)

LanceTable(name='Jokes', version=1, _conn=LanceDBConnection(uri='/Users/efrem_27/Documents/github/AI_engineering/09_lancedb_vector_database/vector_database'))

In [18]:
db.table_names()

['Jokes', 'animals_text']

In [19]:
db.drop_table("jokes")

In [20]:
db.table_names()

['animals_text']

### Open existing table

In [21]:
db.open_table("animals_text").head(2)

pyarrow.Table
text: string
vector: fixed_size_list<item: float>[3]
  child 0, item: float
----
text: [["A small brown dog running.","A cat resting quietly on a sofa."]]
vector: [[[0.12,0.85,0.33],[0.4,0.91,0.1]]]

### Vector search in LanceDB

In [22]:
table_name.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [23]:
query_vector = [0.5, 0.2, 0.9]

table_name.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]",0.2413


### Embedding API

In [24]:

from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get('gemini-text').create(name = "gemini-embedding-001")
model

GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [25]:
hello_embedding = model.generate_embeddings("Hello")

In [26]:
import numpy as np

hello_embedding = np.array(hello_embedding)
hello_embedding.shape

(5, 3072)

In [ ]:
# Creating table here
class JokeModel(LanceModel):
    joke: str = model.SourceField()
    vector: Vector(3072) = model.VectorField()

table_jokes = db.create_table('jokes', schema = JokeModel, exist_ok=True)
table_jokes

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='/Users/efrem_27/Documents/github/AI_engineering/09_lancedb_vector_database/vector_database'))

In [31]:
import pandas as pd

with open("data/jokes.json", "r") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


In [32]:
df_jokes["joke"].iloc[0]

'Parallel lines have so much in common—it’s sad they’ll never meet.'

##### add data to table

In [33]:
table_jokes.head()

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[3072]
  child 0, item: float
----
joke: []
vector: []

In [ ]:
table_jokes.add(df_jokes)

### Perform vector search

In [46]:
table_jokes.search("Data engineering jokes").limit(5).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.460492
1,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.460492
2,"Data engineer motto: If it works, don’t touch ...","[-0.020296954, 0.020327171, -0.009069326, -0.0...",0.542574
3,"Data engineer motto: If it works, don’t touch ...","[-0.020296954, 0.020327171, -0.009069326, -0.0...",0.542574
4,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.656061


In [47]:
table_jokes.search("Give me some chemistry jokes").limit(5).to_pandas()

,joke,vector,_distance
0,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.582086
1,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.582086
2,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.617609
3,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.617609
4,Chemists have all the solutions… mostly in bea...,"[-0.012411982, -0.002532549, -0.0061755716, -0...",0.679975


In [48]:
table_jokes.search("Give me some nature jokes").limit(5).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.615796
1,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.615796
2,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.711001
3,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.711001
4,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.729067


### Hybrid search

In [49]:
table_jokes.create_fts_index("joke", replace=True)

In [50]:
from lancedb import rerankers

# reranks the results from a vector and full text search
reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "give me nature related jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="joke",
).rerank(reranker).limit(8).to_pandas()

# somewhat different to before
results

,joke,vector,_relevance_score
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.032266
1,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.031754
2,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.031099
3,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.031054
4,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.015873
5,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.015625
6,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.015385
7,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.015152
